# Lab 06 — 01 Gold Dimensions

**Dataset:** Synthea Healthcare  
**Layer:** Gold dimensional model

## Purpose

Build and validate the six conformed dimensions used by the Lab 06 star schema:

- `dim_date`
- `dim_patient`
- `dim_provider`
- `dim_organization`
- `dim_payer`
- `dim_condition`

### Design choices

- Shared paths and fully qualified table names come from `src/config.py`.
- No `current_user()` or workspace-user paths.
- Catalog/schema/volume remain runtime parameters and are passed into `Lab06Config`.
- `dim_date` is generated independently from fact data.
- `dim_date` is generated once by default and reused on later runs.
- Source business identifiers are preserved alongside deterministic surrogate keys.
- Gold target data types are explicitly cast instead of relying on CSV inference.

## 1. Runtime context

Runtime values are supplied by the Databricks Job and loaded centrally from `src/runtime_config.py`. This notebook does not create widgets.

In [0]:
import sys
from pathlib import Path

lab_root = Path.cwd().parent
if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

from src.runtime_config import load_runtime_context

ctx = load_runtime_context(
    dbutils,
    include_date_config=True,
)

catalog = ctx.catalog
schema = ctx.schema
volume_name = ctx.volume_name
config = ctx.config

date_start = ctx.date_start
date_end = ctx.date_end
rebuild_dim_date = ctx.rebuild_dim_date

print(f"Catalog          : {catalog}")
print(f"Schema           : {schema}")
print(f"Volume           : {volume_name}")
print(f"Date range       : {date_start} -> {date_end}")
print(f"Rebuild dim_date : {rebuild_dim_date}")

## 2. Shared helpers

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

volume_path = config.volume_path
reference_path = config.reference_path

table_names = {
    "date": config.dim_date,
    "patient": config.dim_patient,
    "provider": config.dim_provider,
    "organization": config.dim_organization,
    "payer": config.dim_payer,
    "condition": config.dim_condition,
}

print(f"Volume path    : {volume_path}")
print(f"Reference path : {reference_path}")

def read_reference_csv(file_name: str) -> DataFrame:
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(f"{reference_path}/{file_name}")
    )

def overwrite_delta_table(df: DataFrame, table_name: str) -> None:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

def dimension_key_profile(
    table_name: str,
    key_column: str,
) -> tuple[int, int, int]:
    df = spark.table(table_name)
    row = (
        df.agg(
            F.count("*").alias("row_count"),
            F.countDistinct(key_column).alias("distinct_key_count"),
            F.sum(
                F.when(F.col(key_column).isNull(), 1).otherwise(0)
            ).alias("null_key_count"),
        )
        .first()
    )
    return (
        int(row["row_count"]),
        int(row["distinct_key_count"]),
        int(row["null_key_count"] or 0),
    )

## 3. Validate staged reference files

The source-preparation notebook should already have staged these five reference files.

In [0]:
REQUIRED_REFERENCE_FILES = {
    "patients.csv",
    "providers.csv",
    "organizations.csv",
    "payers.csv",
    "conditions.csv",
}

available_reference_files = {
    item.name.rstrip("/")
    for item in dbutils.fs.ls(reference_path)
}

missing_reference_files = sorted(
    REQUIRED_REFERENCE_FILES - available_reference_files
)

reference_validation_df = spark.createDataFrame(
    [
        (
            file_name,
            "PASS" if file_name in available_reference_files else "FAIL",
        )
        for file_name in sorted(REQUIRED_REFERENCE_FILES)
    ],
    ["file_name", "status"],
)

display(reference_validation_df)

if missing_reference_files:
    raise FileNotFoundError(
        "Missing staged reference files: "
        + ", ".join(missing_reference_files)
    )

print("Reference source validation passed.")

## 4. Build `dim_date`

`dim_date` is **not derived from encounters or another fact table**.

A complete calendar is generated independently so:

- days with zero healthcare activity remain available for reporting,
- future facts can reuse the same calendar,
- the dimension does not require repeatedly scanning fact data.

By default, an existing `dim_date` is reused. Set `rebuild_dim_date=true` only when the configured calendar range needs to change.

In [0]:
dim_date_exists = spark.catalog.tableExists(table_names["date"])

if (not dim_date_exists) or rebuild_dim_date:
    date_seed_df = spark.createDataFrame(
        [(date_start, date_end)],
        ["start_date", "end_date"],
    )

    dim_date_df = (
        date_seed_df
        .select(
            F.explode(
                F.sequence(
                    F.to_date("start_date"),
                    F.to_date("end_date"),
                    F.expr("INTERVAL 1 DAY"),
                )
            ).alias("full_date")
        )
        .select(
            F.date_format("full_date", "yyyyMMdd")
                .cast("int")
                .alias("date_key"),
            F.col("full_date"),
            F.year("full_date").alias("year"),
            F.quarter("full_date").alias("quarter"),
            F.month("full_date").alias("month"),
            F.date_format("full_date", "MMMM").alias("month_name"),
            F.weekofyear("full_date").alias("week_of_year"),
            F.dayofmonth("full_date").alias("day_of_month"),
            F.date_format("full_date", "EEEE").alias("day_name"),
            (
                F.pmod(F.dayofweek("full_date") + F.lit(5), F.lit(7))
                + F.lit(1)
            ).alias("iso_day_of_week"),
            F.date_format("full_date", "yyyy-MM").alias("year_month"),
            F.when(
                F.dayofweek("full_date").isin(1, 7),
                F.lit(True),
            )
            .otherwise(F.lit(False))
            .alias("is_weekend"),
        )
    )

    overwrite_delta_table(
        dim_date_df,
        table_names["date"],
    )

    print(
        f"dim_date generated for {date_start} -> {date_end}"
    )
else:
    print(
        "dim_date already exists. Reusing it without regeneration."
    )

display(
    spark.table(table_names["date"])
    .orderBy("full_date")
    .limit(10)
)

## 5. Build `dim_patient`

Patient-identifying attributes are intentionally retained because they will later be used to demonstrate **Column-Level Security** on synthetic data.

In [0]:
patients_src = read_reference_csv("patients.csv")

dim_patient_df = (
    patients_src
    .select(
        F.xxhash64("Id").alias("patient_key"),
        F.col("Id").alias("patient_id"),
        F.to_date("BIRTHDATE").alias("birth_date"),
        F.to_date("DEATHDATE").alias("death_date"),
        F.col("SSN").alias("ssn"),
        F.col("FIRST").alias("first_name"),
        F.col("LAST").alias("last_name"),
        F.col("MAIDEN").alias("maiden_name"),
        F.col("MARITAL").alias("marital_status"),
        F.col("RACE").alias("race"),
        F.col("ETHNICITY").alias("ethnicity"),
        F.col("GENDER").alias("gender"),
        F.col("BIRTHPLACE").alias("birthplace"),
        F.col("ADDRESS").alias("address"),
        F.col("CITY").alias("city"),
        F.col("STATE").alias("state"),
        F.col("COUNTY").alias("county"),
        F.col("ZIP").alias("zip_code"),
        F.col("LAT").cast("double").alias("latitude"),
        F.col("LON").cast("double").alias("longitude"),
        F.col("HEALTHCARE_EXPENSES")
            .cast("decimal(18,2)")
            .alias("healthcare_expenses"),
        F.col("HEALTHCARE_COVERAGE")
            .cast("decimal(18,2)")
            .alias("healthcare_coverage"),
    )
    .dropDuplicates(["patient_id"])
)

overwrite_delta_table(
    dim_patient_df,
    table_names["patient"],
)

print(f"Built {table_names['patient']}")

## 6. Build `dim_organization`

`organization_id` will later be the primary RLS scoping attribute for encounter analytics.

In [0]:
organizations_src = read_reference_csv("organizations.csv")

dim_organization_df = (
    organizations_src
    .select(
        F.xxhash64("Id").alias("organization_key"),
        F.col("Id").alias("organization_id"),
        F.col("NAME").alias("organization_name"),
        F.col("ADDRESS").alias("address"),
        F.col("CITY").alias("city"),
        F.col("STATE").alias("state"),
        F.col("ZIP").alias("zip_code"),
        F.col("LAT").cast("double").alias("latitude"),
        F.col("LON").cast("double").alias("longitude"),
        F.col("PHONE").alias("phone"),
        F.col("REVENUE")
            .cast("decimal(18,2)")
            .alias("revenue"),
        F.col("UTILIZATION")
            .cast("long")
            .alias("utilization"),
    )
    .dropDuplicates(["organization_id"])
)

overwrite_delta_table(
    dim_organization_df,
    table_names["organization"],
)

print(f"Built {table_names['organization']}")

## 7. Build `dim_provider`

The Synthea source spells the specialty field as `SPECIALITY`; the Gold model standardizes it to `specialty`.

In [0]:
providers_src = read_reference_csv("providers.csv")

dim_provider_df = (
    providers_src
    .select(
        F.xxhash64("Id").alias("provider_key"),
        F.col("Id").alias("provider_id"),
        F.col("ORGANIZATION").alias("organization_id"),
        F.col("NAME").alias("provider_name"),
        F.col("GENDER").alias("gender"),
        F.col("SPECIALITY").alias("specialty"),
        F.col("ADDRESS").alias("address"),
        F.col("CITY").alias("city"),
        F.col("STATE").alias("state"),
        F.col("ZIP").alias("zip_code"),
        F.col("LAT").cast("double").alias("latitude"),
        F.col("LON").cast("double").alias("longitude"),
        F.col("UTILIZATION")
            .cast("long")
            .alias("utilization"),
    )
    .dropDuplicates(["provider_id"])
)

overwrite_delta_table(
    dim_provider_df,
    table_names["provider"],
)

print(f"Built {table_names['provider']}")

## 8. Build `dim_payer`

In [0]:
payers_src = read_reference_csv("payers.csv")

dim_payer_df = (
    payers_src
    .select(
        F.xxhash64("Id").alias("payer_key"),
        F.col("Id").alias("payer_id"),
        F.col("NAME").alias("payer_name"),
        F.col("CITY").alias("city"),
        F.col("STATE_HEADQUARTERED").alias("state_headquartered"),
        F.col("ZIP").alias("zip_code"),
        F.col("PHONE").alias("phone"),
        F.col("AMOUNT_COVERED")
            .cast("decimal(18,2)")
            .alias("amount_covered"),
        F.col("AMOUNT_UNCOVERED")
            .cast("decimal(18,2)")
            .alias("amount_uncovered"),
        F.col("REVENUE")
            .cast("decimal(18,2)")
            .alias("revenue"),
        F.col("COVERED_ENCOUNTERS")
            .cast("long")
            .alias("covered_encounters"),
        F.col("UNCOVERED_ENCOUNTERS")
            .cast("long")
            .alias("uncovered_encounters"),
        F.col("UNIQUE_CUSTOMERS")
            .cast("long")
            .alias("unique_customers"),
        F.col("QOLS_AVG")
            .cast("double")
            .alias("qols_avg"),
        F.col("MEMBER_MONTHS")
            .cast("long")
            .alias("member_months"),
    )
    .dropDuplicates(["payer_id"])
)

overwrite_delta_table(
    dim_payer_df,
    table_names["payer"],
)

print(f"Built {table_names['payer']}")

## 9. Build `dim_condition`

The source contains a small number of condition codes with more than one textual description.  
Therefore the dimension grain is:

**one row per distinct `(condition_code, condition_description)` pair**.

In [0]:
conditions_src = read_reference_csv("conditions.csv")

dim_condition_df = (
    conditions_src
    .select(
        F.col("CODE").alias("condition_code"),
        F.col("DESCRIPTION").alias("condition_description"),
    )
    .dropDuplicates(
        ["condition_code", "condition_description"]
    )
    .withColumn(
        "condition_key",
        F.xxhash64(
            "condition_code",
            "condition_description",
        ),
    )
    .select(
        "condition_key",
        "condition_code",
        "condition_description",
    )
)

overwrite_delta_table(
    dim_condition_df,
    table_names["condition"],
)

print(f"Built {table_names['condition']}")

## 10. Dimension key validation

Every dimension must have:

- non-null surrogate keys,
- unique surrogate keys,
- no duplicate business-grain rows.

In [0]:
DIMENSION_KEYS = {
    "dim_date": (
        table_names["date"],
        "date_key",
    ),
    "dim_patient": (
        table_names["patient"],
        "patient_key",
    ),
    "dim_provider": (
        table_names["provider"],
        "provider_key",
    ),
    "dim_organization": (
        table_names["organization"],
        "organization_key",
    ),
    "dim_payer": (
        table_names["payer"],
        "payer_key",
    ),
    "dim_condition": (
        table_names["condition"],
        "condition_key",
    ),
}

key_validation_rows = []
failed_key_checks = []

for dimension_name, (
    table_name,
    key_column,
) in DIMENSION_KEYS.items():

    (
        row_count,
        distinct_key_count,
        null_key_count,
    ) = dimension_key_profile(
        table_name,
        key_column,
    )

    status = (
        "PASS"
        if (
            row_count == distinct_key_count
            and null_key_count == 0
        )
        else "FAIL"
    )

    key_validation_rows.append(
        (
            dimension_name,
            row_count,
            distinct_key_count,
            null_key_count,
            status,
        )
    )

    if status == "FAIL":
        failed_key_checks.append(dimension_name)

key_validation_df = spark.createDataFrame(
    key_validation_rows,
    [
        "dimension",
        "row_count",
        "distinct_key_count",
        "null_key_count",
        "status",
    ],
)

display(
    key_validation_df.orderBy("dimension")
)

if failed_key_checks:
    raise ValueError(
        "Dimension key validation failed: "
        + ", ".join(failed_key_checks)
    )

print("All dimension key checks passed.")

## 11. Source-to-dimension reconciliation

In [0]:
source_expected_counts = {
    "dim_patient": patients_src.select("Id").distinct().count(),
    "dim_provider": providers_src.select("Id").distinct().count(),
    "dim_organization": organizations_src.select("Id").distinct().count(),
    "dim_payer": payers_src.select("Id").distinct().count(),
    "dim_condition": (
        conditions_src
        .select("CODE", "DESCRIPTION")
        .distinct()
        .count()
    ),
}

reconciliation_rows = []
failed_reconciliation = []

for dimension_name, expected_count in source_expected_counts.items():
    table_name, _ = DIMENSION_KEYS[dimension_name]
    actual_count = spark.table(table_name).count()

    status = (
        "PASS"
        if actual_count == expected_count
        else "FAIL"
    )

    reconciliation_rows.append(
        (
            dimension_name,
            expected_count,
            actual_count,
            status,
        )
    )

    if status == "FAIL":
        failed_reconciliation.append(dimension_name)

reconciliation_df = spark.createDataFrame(
    reconciliation_rows,
    [
        "dimension",
        "expected_source_rows",
        "actual_dimension_rows",
        "status",
    ],
)

display(
    reconciliation_df.orderBy("dimension")
)

if failed_reconciliation:
    raise ValueError(
        "Source-to-dimension reconciliation failed: "
        + ", ".join(failed_reconciliation)
    )

print("Source-to-dimension reconciliation passed.")

## 12. Validate `dim_date` completeness

In [0]:
dim_date_validation = (
    spark.table(table_names["date"])
    .agg(
        F.min("full_date").alias("actual_start"),
        F.max("full_date").alias("actual_end"),
        F.count("*").alias("actual_rows"),
        F.countDistinct("full_date").alias("distinct_dates"),
    )
    .first()
)

expected_date_rows = (
    spark.range(1)
    .select(
        (
            F.datediff(
                F.to_date(F.lit(date_end)),
                F.to_date(F.lit(date_start)),
            )
            + F.lit(1)
        ).alias("expected_rows")
    )
    .first()["expected_rows"]
)

date_status = (
    "PASS"
    if (
        dim_date_validation["actual_rows"] == expected_date_rows
        and dim_date_validation["distinct_dates"] == expected_date_rows
        and str(dim_date_validation["actual_start"]) == date_start
        and str(dim_date_validation["actual_end"]) == date_end
    )
    else "FAIL"
)

date_validation_df = spark.createDataFrame(
    [
        (
            date_start,
            date_end,
            int(expected_date_rows),
            str(dim_date_validation["actual_start"]),
            str(dim_date_validation["actual_end"]),
            int(dim_date_validation["actual_rows"]),
            date_status,
        )
    ],
    [
        "expected_start",
        "expected_end",
        "expected_rows",
        "actual_start",
        "actual_end",
        "actual_rows",
        "status",
    ],
)

display(date_validation_df)

if date_status == "FAIL":
    raise ValueError(
        "dim_date does not match the configured complete calendar range."
    )

print("dim_date completeness validation passed.")

## 13. Validate provider → organization relationship

Every populated provider organization reference should resolve to `dim_organization`.

In [0]:
provider_organization_orphans = (
    spark.table(table_names["provider"])
    .alias("p")
    .filter(F.col("p.organization_id").isNotNull())
    .join(
        spark.table(table_names["organization"])
        .select("organization_id")
        .alias("o"),
        F.col("p.organization_id")
        == F.col("o.organization_id"),
        "left_anti",
    )
    .count()
)

relationship_validation_df = spark.createDataFrame(
    [
        (
            "dim_provider -> dim_organization",
            provider_organization_orphans,
            (
                "PASS"
                if provider_organization_orphans == 0
                else "FAIL"
            ),
        )
    ],
    [
        "relationship",
        "orphan_rows",
        "status",
    ],
)

display(relationship_validation_df)

if provider_organization_orphans:
    raise ValueError(
        f"Found {provider_organization_orphans} provider rows "
        "with unresolved organization references."
    )

print("Provider-to-organization relationship validation passed.")

## 14. Gold dimension inventory

In [0]:
dimension_inventory = []

for dimension_name, (
    table_name,
    key_column,
) in DIMENSION_KEYS.items():

    df = spark.table(table_name)

    dimension_inventory.append(
        (
            dimension_name,
            table_name,
            df.count(),
            len(df.columns),
            key_column,
        )
    )

dimension_inventory_df = spark.createDataFrame(
    dimension_inventory,
    [
        "dimension",
        "table_name",
        "row_count",
        "column_count",
        "surrogate_key",
    ],
)

display(
    dimension_inventory_df.orderBy("dimension")
)

## 15. Sample Gold dimensions

In [0]:
display(
    spark.table(table_names["patient"])
    .select(
        "patient_key",
        "patient_id",
        "birth_date",
        "gender",
        "race",
        "state",
    )
    .limit(10)
)

In [0]:
display(
    spark.table(table_names["organization"])
    .select(
        "organization_key",
        "organization_id",
        "organization_name",
        "city",
        "state",
        "revenue",
    )
    .limit(10)
)

In [0]:
display(
    spark.table(table_names["condition"])
    .orderBy("condition_code", "condition_description")
    .limit(20)
)

## 16. Completion

Gold dimensions created:

```text
dim_date
dim_patient
dim_provider
dim_organization
dim_payer
dim_condition
```

### Dimension grains

| Dimension | Grain |
|---|---|
| `dim_date` | one row per calendar date |
| `dim_patient` | one row per patient |
| `dim_provider` | one row per provider |
| `dim_organization` | one row per healthcare organization |
| `dim_payer` | one row per payer |
| `dim_condition` | one row per distinct condition code + description |

**Next:** build `fact_encounters`.

In [0]:
print("LAB 06 — GOLD DIMENSIONS COMPLETE")
print("")
print("Created:")
for name in [
    "dim_date",
    "dim_patient",
    "dim_provider",
    "dim_organization",
    "dim_payer",
    "dim_condition",
]:
    print(f"  - {catalog}.{schema}.{name}")

print("")
print("Next: lab06_02_fact_encounters")